# 📚 Taller 5: El Asistente del Decano — RAG sobre el reglamento

En la **Universidad Semillero** existe una sola persona que se sabe el reglamento de memoria: **Don Carlos**, el secretario académico. Cada día su oficina recibe la misma fila de estudiantes preguntando lo mismo:

> *"¿Cuántas faltas puedo tener?"*  
> *"¿Hasta cuándo puedo retirar una materia?"*  
> *"¿Quién conforma el tribunal de tesis?"*

Don Carlos quiere jubilarse. **Tu misión:** construirle un asistente con IA que responda exactamente como él — citando el artículo correcto, sin inventar.

Pero hay un problema: si le preguntas a un LLM tal cual sobre el reglamento de la Universidad Semillero, va a **alucinar** una respuesta plausible pero falsa. El modelo nunca vio ese reglamento.

La solución es **RAG** (*Retrieval Augmented Generation*): mantener al modelo intacto y darle el reglamento como una biblioteca consultable.

### Lo que vas a construir

| Fase | Paso | Qué hace |
|---|---|---|
| **Indexación** (offline) | Documentos + chunking | Cortar el reglamento en pedazos manejables |
| **Indexación** (offline) | Embeddings + ChromaDB | Vectorizar cada chunk y guardarlo |
| **Consulta** (online) | Pregunta + búsqueda | Encontrar los chunks más relevantes |
| **Consulta** (online) | Contexto + respuesta | El LLM responde apoyado en lo recuperado |

⏱️ **Tiempo estimado:** 45 minutos

> ⚠️ **Antes de empezar:** asegúrate de tener Ollama corriendo (`ollama serve` en otra terminal). La siguiente celda descargará los modelos que usaremos.

In [ ]:
# 1. Instalamos las librerias
!pip install ollama chromadb -q

# 2. Descargamos los modelos (Ollama omite la descarga si ya los tienes)
!ollama pull nomic-embed-text
!ollama pull gemma4:e2b

## 1️⃣ Carga del documento

Don Carlos nos entregó el reglamento en un archivo `.txt`. Vamos a cargarlo y echarle un vistazo.

In [ ]:
with open("reglamento_universidad_semillero.txt", "r", encoding="utf-8") as f:
    reglamento = f.read()

print(f"Caracteres totales: {len(reglamento):,}")
print(f"Palabras totales:   {len(reglamento.split()):,}")
print(f"\nPrimeros 400 caracteres:\n{'-'*60}")
print(reglamento[:400])

## 2️⃣ Chunking — Cortar el reglamento en pedazos

Un LLM no puede recibir todo el reglamento como contexto en cada pregunta — sería caro y lento. Y la base vectorial necesita pedazos pequeños para que la búsqueda sea precisa.

**La idea:** cortar el documento en chunks de tamaño manejable (la slide sugiere ~200-500 tokens, equivalente a 1-2 párrafos). Cada chunk se vectoriza por separado.

**Estrategia para este documento:** un reglamento legal tiene una estructura muy clara — cada artículo es una unidad semántica completa que habla de un solo tema. Si cortamos por caracteres podemos partir un artículo por la mitad o mezclar dos artículos en un mismo chunk, lo que confunde al embedding.

La mejor estrategia aquí es **un chunk por artículo**: cada artículo con todos sus párrafos vive en su propio chunk. Esto hace que el embedding capture exactamente el tema de ese artículo y la búsqueda sea precisa.

> 💡 En documentos sin estructura clara (libros, transcripciones, blogs) sí conviene cortar por tamaño con overlap. La estrategia depende siempre de la fuente.

In [ ]:
import re

def chunkear_por_articulo(texto):
    """Corta el texto en chunks: un articulo por chunk.

    Localizamos cada cabecera 'Articulo N.' con finditer y rebanamos
    el texto entre cabeceras consecutivas. Cada chunk arranca exacto
    en su 'Articulo N.' y termina justo antes del siguiente.
    """
    cabeceras = list(re.finditer(r"^Articulo \d+\.", texto, flags=re.MULTILINE))

    chunks = []
    for i, m in enumerate(cabeceras):
        ini = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        parte = texto[ini:fin]
        # Si entre articulos cae un titulo de capitulo "CAPITULO X", lo recortamos
        limpio = re.split(r"\n\s*\n\s*CAPITULO", parte)[0].strip()
        chunks.append(limpio)
    return chunks

chunks = chunkear_por_articulo(reglamento)

print(f"Total de chunks: {len(chunks)}")
print(f"Tamano promedio: {sum(len(c) for c in chunks) // len(chunks)} caracteres")
print(f"Tamano min/max:  {min(len(c) for c in chunks)} / {max(len(c) for c in chunks)} caracteres\n")

# Verificacion: cada chunk DEBE empezar con 'Articulo N.'
print("Inicio de cada chunk:")
print("-" * 60)
for i, c in enumerate(chunks):
    primeros = c[:55].replace("\n", " ")
    print(f"  ch_{i:02d}: {primeros}...")

## 3️⃣ Embeddings + ChromaDB — Indexar los chunks

Cada chunk se transforma en un vector de 768 dimensiones con `nomic-embed-text`, y guardamos esos vectores en **ChromaDB** — una base de datos vectorial que sabe buscar por similitud.

**Dos detalles importantes:**

1. **Prefijos de `nomic-embed-text`.** Este modelo fue entrenado con dos modos: `search_document:` para indexar y `search_query:` para consultar. Sin esos prefijos, los embeddings de chunks y de preguntas viven en espacios desalineados y la búsqueda falla.
2. **Métrica de distancia coseno.** ChromaDB usa L2 por defecto, pero para embeddings de texto la **similitud coseno** funciona mucho mejor. La activamos al crear la colección con `metadata={"hnsw:space": "cosine"}`.

**¿Por qué una base vectorial y no una lista en NumPy?**  
Para 33 chunks daría igual. Pero ChromaDB:
- Guarda los vectores en disco (o memoria) sin que tengamos que recalcularlos.
- Indexa para búsqueda rápida cuando hay miles o millones de chunks.
- Asocia cada vector con su texto original y metadata.

In [ ]:
import ollama
import chromadb

MODELO_EMBEDDING = "nomic-embed-text"

def vectorizar_documento(texto):
    """Embedding de un chunk para indexar (con prefijo de documento)."""
    return ollama.embeddings(
        model=MODELO_EMBEDDING,
        prompt=f"search_document: {texto}",
    )["embedding"]

def vectorizar_consulta(texto):
    """Embedding de una pregunta para buscar (con prefijo de query)."""
    return ollama.embeddings(
        model=MODELO_EMBEDDING,
        prompt=f"search_query: {texto}",
    )["embedding"]

# Cliente de Chroma en memoria (para persistencia: chromadb.PersistentClient(path="./chroma_db"))
cliente = chromadb.Client()

# Si ya existe la coleccion la borramos para empezar limpio
try:
    cliente.delete_collection("reglamento")
except Exception:
    pass

# Creamos la coleccion usando similitud COSENO en vez de L2 (mejor para texto)
coleccion = cliente.create_collection(
    name="reglamento",
    metadata={"hnsw:space": "cosine"},
)

# Vectorizamos e insertamos cada chunk (con prefijo de documento)
print("Indexando chunks en ChromaDB...\n")
for i, chunk in enumerate(chunks):
    vector = vectorizar_documento(chunk)
    coleccion.add(
        ids=[f"ch_{i:02d}"],
        embeddings=[vector],
        documents=[chunk],
        metadatas=[{"chunk_n": i, "fuente": "reglamento_universidad_semillero.txt"}],
    )
    print(f"  [OK] ch_{i:02d}  ({len(chunk)} chars)")

print(f"\n{coleccion.count()} chunks indexados en la coleccion '{coleccion.name}'.")

## 4️⃣ Pregunta + Búsqueda — Recuperar lo relevante

Cuando un estudiante hace una pregunta, vectorizamos la pregunta **con el mismo modelo** y le pedimos a ChromaDB los chunks más parecidos.

El parámetro `top_k` controla cuántos chunks recuperamos. Pocos = arriesgamos perder el chunk correcto. Muchos = metemos ruido al prompt.

In [ ]:
def buscar(pregunta, top_k=3):
    """Devuelve los top_k chunks mas relevantes para la pregunta.

    Usamos el prefijo 'search_query:' para alinear el embedding de la
    pregunta con los embeddings de documento que indexamos antes.
    """
    v_pregunta = vectorizar_consulta(pregunta)
    resultado = coleccion.query(query_embeddings=[v_pregunta], n_results=top_k)
    return list(zip(resultado["ids"][0], resultado["documents"][0], resultado["distances"][0]))

pregunta = "Cuantas faltas puedo tener antes de perder una materia?"
resultados = buscar(pregunta, top_k=3)

print(f"Pregunta: {pregunta}\n")
print("Top 3 chunks recuperados:")
print("=" * 70)
for chunk_id, texto, distancia in resultados:
    print(f"\n  [{chunk_id}]  distancia: {distancia:.3f}")
    print(f"  {texto[:250]}...")

## 5️⃣ Contexto + Respuesta — Generar con el LLM

Ya tenemos los chunks relevantes. Ahora los **inyectamos en el prompt** del LLM junto con la pregunta y una instrucción clara: *responde solo con base en el contexto*.

Esto es lo que hace que la respuesta esté **anclada (grounded)** al reglamento real, no a lo que el modelo "recuerde" de su entrenamiento.

In [ ]:
MODELO_LLM = "gemma4:e2b"

PROMPT_SISTEMA = """Eres el asistente academico de la Universidad Semillero. Respondes preguntas sobre el reglamento institucional.

Reglas estrictas:
- Responde UNICAMENTE con base en el CONTEXTO entregado.
- Cita el numero de articulo en tu respuesta cuando sea posible.
- Si la informacion no esta en el contexto, responde exactamente: "No tengo esa informacion en el reglamento."
- Se breve y directo. No inventes datos."""

def responder_con_rag(pregunta, top_k=3, mostrar_contexto=False):
    """Pipeline RAG completo: busca chunks, arma el prompt, genera respuesta."""
    # 1. Recuperar contexto
    chunks_recuperados = buscar(pregunta, top_k=top_k)
    contexto = "\n\n---\n\n".join(texto for _, texto, _ in chunks_recuperados)

    # 2. Armar prompt y generar
    respuesta = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"},
        ],
    )
    texto_respuesta = respuesta["message"]["content"]

    if mostrar_contexto:
        print("CONTEXTO RECUPERADO:")
        print("-" * 70)
        for cid, txt, dist in chunks_recuperados:
            print(f"  [{cid}] dist={dist:.3f}: {txt[:120]}...")
        print()

    return texto_respuesta

# Probemos
pregunta = "Cuantas inasistencias puedo tener antes de perder una materia?"
respuesta = responder_con_rag(pregunta, mostrar_contexto=True)

print(f"PREGUNTA: {pregunta}\n")
print("RESPUESTA DEL ASISTENTE:")
print("=" * 70)
print(respuesta)

## 6️⃣ Con RAG vs Sin RAG — La prueba reina

Hagamos la misma pregunta de dos formas:
1. **Sin RAG:** preguntándole directamente al LLM, sin contexto.
2. **Con RAG:** usando nuestro pipeline.

Si el modelo no tiene el reglamento en su entrenamiento (no lo tiene), va a **alucinar** una respuesta plausible. Compárala con la respuesta anclada.

In [ ]:
def responder_sin_rag(pregunta):
    """Pregunta directa al LLM sin contexto adicional."""
    respuesta = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": "Eres un asistente academico de la Universidad Semillero. Responde preguntas del reglamento institucional."},
            {"role": "user", "content": pregunta},
        ],
    )
    return respuesta["message"]["content"]

preguntas_prueba = [
    "Cuantas veces puedo matricularme en la misma materia?",
    "Quien conforma el tribunal de defensa de tesis y cual es la nota minima?",
    "Que pasa si pierdo un libro de la biblioteca?",
]

for q in preguntas_prueba:
    print("=" * 70)
    print(f"PREGUNTA: {q}\n")

    print("[SIN RAG] (el modelo improvisa):")
    print("-" * 70)
    print(responder_sin_rag(q))
    print()

    print("[CON RAG] (el modelo cita el reglamento real):")
    print("-" * 70)
    print(responder_con_rag(q))
    print("\n")

## 7️⃣ El caso del "no tengo esa información"

Un buen asistente RAG **sabe cuándo no sabe**. Si la pregunta no se puede responder con el reglamento, el modelo debe decirlo en lugar de inventar.

In [ ]:
pregunta_fuera_de_alcance = "Cual es el costo de la matricula para el proximo semestre?"

print(f"PREGUNTA: {pregunta_fuera_de_alcance}\n")
print("RESPUESTA:")
print("-" * 70)
print(responder_con_rag(pregunta_fuera_de_alcance))

## 📋 Resumen

| Concepto | Qué hace |
|---|---|
| **Chunking** | Cortar el documento respetando su estructura (un artículo = un chunk) |
| **Embeddings** | Cada chunk se vuelve un vector que captura su significado |
| **ChromaDB** | Base de datos que guarda vectores y busca por similitud |
| **Retrieval** | Vectorizar la pregunta y traer los chunks más cercanos |
| **Augmented prompt** | Inyectar los chunks como contexto del LLM |
| **Generation** | El LLM redacta una respuesta anclada al contexto |

### Lo que hace que RAG funcione (y falle)

✅ **Funciona bien cuando:**
- Los chunks respetan la estructura natural del documento.
- El prompt instruye al modelo a no salirse del contexto.
- La fuente original está limpia y bien estructurada.

❌ **Falla cuando:**
- Los chunks parten ideas a la mitad.
- El prompt no es claro y el modelo "agrega" lo que sabe del entrenamiento.
- La pregunta usa palabras muy distintas a las del documento (ej. "faltas" vs "inasistencias") y la pregunta requiere razonar sobre múltiples chunks que el retrieval no priorizó.

---

## 🎯 Tarea

Don Carlos te dejó tres encargos antes de irse:

### Parte A — Tus propias preguntas

Escribe **5 preguntas** sobre el reglamento de la Universidad Semillero:
- 3 preguntas que **sí** se respondan con el reglamento (de capítulos distintos).
- 2 preguntas que **no** se respondan con el reglamento (deberían disparar el *"No tengo esa información"*).

Ejecuta cada una con `responder_con_rag(...)` y comenta brevemente debajo si la respuesta fue correcta y si citó el artículo adecuado.

### Parte B — Experimenta con `top_k`

Cambia `top_k` a `1`, `3` y `8`. Vuelve a hacer una pregunta compleja como *"¿qué requisitos necesito para titularme?"* (que combina varios artículos del Capítulo VI) y observa cómo cambia la respuesta.

Responde por escrito:
1. ¿`top_k=1` te alcanzó para responder bien? ¿Por qué?
2. ¿`top_k=8` mejoró la respuesta o introdujo ruido?
3. ¿Qué valor te dio el mejor balance?

### Parte C — Agrega una fuente nueva

Crea un segundo archivo `.txt` con un **calendario académico ficticio** (al menos 10 fechas: inicio de clases, exámenes, retiros, matrícula extraordinaria, etc.). Indéxalo en la **misma colección** de ChromaDB con `metadatas={"fuente": "calendario_academico.txt"}`. Luego haz una pregunta que requiera información de **ambos** documentos, por ejemplo: *"¿Hasta qué fecha puedo retirar una materia y cuál es la regla para hacerlo?"*

---

📚 **Don Carlos te observa con orgullo desde su jubilación.**

In [ ]:
# Espacio para tu Parte A - tus 5 preguntas

mis_preguntas = [
    # 3 preguntas que SI se responden con el reglamento
    "",
    "",
    "",
    # 2 preguntas que NO se responden con el reglamento
    "",
    "",
]

# Descomenta cuando hayas escrito tus preguntas:
# for q in mis_preguntas:
#     if not q:
#         continue
#     print("=" * 70)
#     print(f"PREGUNTA: {q}\n")
#     print(responder_con_rag(q))
#     print()

In [ ]:
# Espacio para tu Parte B - experimenta con top_k

# pregunta_compleja = "Que requisitos necesito para titularme?"
# print("=== top_k=1 ===")
# print(responder_con_rag(pregunta_compleja, top_k=1))
# print("\n=== top_k=3 ===")
# print(responder_con_rag(pregunta_compleja, top_k=3))
# print("\n=== top_k=8 ===")
# print(responder_con_rag(pregunta_compleja, top_k=8))